## Experiment1: 1000 SAMPLE Heston MLP baseline

In [4]:
%pip install -q QuantLib

In [5]:
import argparse
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import qmc
from sklearn.preprocessing import StandardScaler
from torch import nn
import QuantLib as ql

In [6]:
FEATURES = ['S0', 'K', 'T', 'r', 'v0', 'kappa', 'theta', 'sigma', 'rho']
BOUNDS = {
    'S0':(0.6, 1.4),
    'T': (0.1, 1.4),
    'r': (0.0, 0.1),
    'v0': (0.05, 0.5),
    'kappa': (0.0, 2.0),
    'theta': (0.0, 0.5),
    'sigma': (0.0, 0.5),
    'rho': (-0.95, 0.0)
}
VARIABLE_FEATURES = list(BOUNDS)
K_FIXED, Q_FIXED, SEED = 1.0, 0.0, 42
TRAIN_N, VALIDATION_N, TEST_N = 1000, 200, 500
OUTPUT_DIR = Path('experiment_baseline')
DATA_CSV = None
EPOCHS, PATIENCE, BATCH_SIZE, LEARNING_RATE = 1000, 100, 64, 1e-3

In [7]:
def sample_parameters(n, seed):
    unit = qmc.LatinHypercube(d=len(VARIABLE_FEATURES), seed=seed).random(n)
    lo = np.array([BOUNDS[k][0] for k in VARIABLE_FEATURES])
    hi = np.array([BOUNDS[k][1] for k in VARIABLE_FEATURES])
    df = pd.DataFrame(qmc.scale(unit, lo, hi), columns=VARIABLE_FEATURES)
    df['K'] = K_FIXED
    df['T'] = np.maximum(1, np.round(df['T'] * 365).astype(int))/365.0
    return df[FEATURES]

In [8]:
def heston_reference_price(row):
    valuation = ql.Date(2, ql.January, 2026)
    ql.Settings.instance().evaluationDate = valuation
    day_count = ql.Actual365Fixed()
    maturity = valuation + max(1, int(round(float(row['T']) * 365)))
    rf = ql.YieldTermStructureHandle(ql.FlatForward(valuation, float(row['r']), day_count))
    div = ql.YieldTermStructureHandle(ql.FlatForward(valuation, Q_FIXED, day_count))
    spot = ql.QuoteHandle(ql.SimpleQuote(float(row['S0'])))
    process = ql.HestonProcess(rf, div, spot, float(row['v0']), float(row['kappa']), float(row['theta']), float(row['sigma']), float(row['rho']))
    model = ql.HestonModel(process)
    option = ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call, float(row['K'])), ql.EuropeanExercise(maturity))
    option.setPricingEngine(ql.AnalyticHestonEngine(model, 128))
    return float(option.NPV())

In [9]:
def create_data(n, seed):
    df = sample_parameters(n,seed)
    prices = []
    df['price'] = [heston_reference_price(row) for row in df.to_dict('records')]
    return df

In [10]:
def check_data(df):
    required = FEATURES + ['price']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Missing columns: {missing}')
    if not np.isfinite(df[required].to_numpy(dtype=float)).all():
        raise ValueError('Non-finite data detected')
    if (df['price'] < -1e-7).any():
        raise ValueError('Materially negative reference prices detected')
    lower = np.maximum(df['S0'] - df['K'] * np.exp(-df['r'] * df['T']),0.0)
    if (df['price'] < lower - 1e-4).any() or (df['price'] > df['S0'] + 1e-4).any():
        raise ValueError('Option prices violate no-arbitrage bounds')
    if df[FEATURES].duplicated().any():
        raise ValueError("Duplicate parameter combinations in a dataset")


In [11]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        width = len(FEATURES)
        for _ in range(3):
            layers.extend([nn.Linear(width, 128), nn.ReLU()])
            width = 128
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
    def forward(self,x):
        return self.net(x)

In [12]:
def metrics(actual, pred):
    error = pred - actual
    return {
        'mae': float(np.mean(np.abs(error))),
        'rmse': float(np.sqrt(np.mean(error**2))),
        'median_absolute_error': float(np.median(np.abs(error))),
        'p90_absolute_error': float(np.percentile(np.abs(error), 90)),
        'max_absolute_error': float(np.max(np.abs(error))),
        'relative_error_percent_on_prices_above_0_01': float(
            100 * np.mean(np.abs(error[actual >= 0.01])/actual[actual >= 0.01])
        )if np.any(actual >= 0.01) else None,
    }

In [13]:
def main():
    out = OUTPUT_DIR
    out.mkdir(parents=True, exist_ok=True)
    seed=42
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(min(4, torch.get_num_threads()))

    if DATA_CSV is None:
        print('Generating training/validation/test reference prices...')
        train = create_data(TRAIN_N, 42)
        val = create_data(VALIDATION_N, 43)
        test = create_data(TEST_N, 44)
    else:
        df = pd.read_csv(DATA_CSV)
        if len(df) < 1700:
            raise ValueError('Need at least 1700 rows: 1000 train, 200 val, 500 test')
        df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
        train, val, test = df.iloc[:1000].copy(), df.iloc[1000:1200].copy(), df.iloc[1200:1700].copy()
    for name, frame in [('train', train), ('validation', val), ('test', test)]:
        check_data(frame)
        frame.to_csv(out/f'{name}.csv', index=False)
    xs = StandardScaler().fit(train[FEATURES].to_numpy())
    ys = StandardScaler().fit(train[['price']].to_numpy())

    def tensor_data(frame):
        x = torch.tensor(xs.transform(frame[FEATURES].to_numpy()), dtype=torch.float32)
        y = torch.tensor(ys.transform(frame[['price']].to_numpy()), dtype=torch.float32)
        return x,y

    xtr, ytr = tensor_data(train)
    xva, yva= tensor_data(val)
    xte, _ = tensor_data(test)
    net = MLP()
    optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()
    best_loss = float('inf')
    best_state = None
    stale = 0
    history = []
    start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        net.train()
        perm = torch.randperm(len(xtr))
        for indices in perm.split(BATCH_SIZE):
            optimizer.zero_grad()
            loss = criterion(net(xtr[indices]), ytr[indices])
            loss.backward()
            optimizer.step()
        net.eval()
        with torch.inference_mode():
            train_loss = criterion(net(xtr), ytr).item()
            val_loss = criterion(net(xva), yva).item()
        history.append({
            'epoch': epoch,
            'train_mse_scaled': train_loss,
            'validation_mse_scaled': val_loss
        })
        if val_loss < best_loss - 1e-7:
            best_loss = val_loss
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
            best_epoch = epoch
            stale = 0
        else:
            stale += 1
        if epoch % 50 == 0:
            print(f'Epoch {epoch:4d}: train={train_loss:.6f} val={val_loss:.6f}')
        if stale >= PATIENCE:
            break
    training_seconds = time.perf_counter() - start
    net.load_state_dict(best_state)
    net.eval()
    with torch.inference_mode():
        pred = ys.inverse_transform(net(xte).numpy()).ravel()
    actual = test['price'].to_numpy()
    result = metrics(actual, pred)
    result.update({
        'train_n':1000,
        'validation_n': 200,
        'test_n': 500,
        'best_epoch': best_epoch,
        'training_seconds': training_seconds,
        'parameter_count': sum(p.numel() for p in net.parameters())
    })

    with torch.inference_mode():
        for _ in range(20):
            net(xte)
        timings = []
        for _ in range(100):
            t0 = time.perf_counter()
            net(xte)
            timings.append(time.perf_counter() - t0)
    result['median_batch_inference_ms_500'] = 1000 * float(np.median(timings))
    result['median_inference_microseconds_per_option_batched'] = (
        1e6 * float(np.median(timings))/len(test)
    )
    pd.DataFrame(history).to_csv(out / 'training_history.csv', index=False)
    test.assign(predicted_price = pred, absolute_error = np.abs(pred-actual)).to_csv(out/'test_predictions.csv', index=False)
    torch.save({
        'model_state_dict': net.state_dict(),
        'features': FEATURES,
        'x_mean': xs.mean_,
        'x_scale': xs.scale_,
        'y_mean': ys.mean_,
        'y_scale': ys.scale_,
        'bounds': BOUNDS,
        'seed': seed
    }, out / 'model.pt')
    (out/'metrics.json').write_text(json.dumps(result, indent=2))
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots()
    ax.plot([h['epoch'] for h in history], [h['train_mse_scaled'] for h in history], label='Train')
    ax.plot([h['epoch'] for h in history], [h['validation_mse_scaled'] for h in history], label='Validation')
    ax.set(xlabel='Epoch',ylabel='Scaled MSE', title='MLP learning curves')
    ax.legend()
    fig.tight_layout()
    fig.savefig(out/'learning_curves.png', dpi=180)
    plt.close(fig)
    fig, ax = plt.subplots()
    ax.scatter(actual, pred, alpha=0.5, s=14)
    low, high = min(actual.min(), pred.min()), max(actual.max(), pred.max())
    ax.plot([low, high], [low, high], '--')
    ax.set(xlabel='Heston reference price', ylabel='MLP prediction', title='Unseen test predictions')
    fig.tight_layout()
    fig.savefig(out/'predicted_vs_reference.png', dpi=180)
    plt.close(fig)
    print(json.dumps(result, indent=2))
    print(f'Files saved to {out.resolve()}')

In [14]:
main()

Generating training/validation/test reference prices...
Epoch   50: train=0.001093 val=0.003259
Epoch  100: train=0.000078 val=0.002266
Epoch  150: train=0.000447 val=0.002367
Epoch  200: train=0.000131 val=0.002189
Epoch  250: train=0.000315 val=0.002117
Epoch  300: train=0.000035 val=0.001933
Epoch  350: train=0.000011 val=0.001849
Epoch  400: train=0.000130 val=0.002026
Epoch  450: train=0.000459 val=0.001942
Epoch  500: train=0.000011 val=0.001702
Epoch  550: train=0.000008 val=0.001633
{
  "mae": 0.0047633700045985276,
  "rmse": 0.006298666540378026,
  "median_absolute_error": 0.0037896799518902305,
  "p90_absolute_error": 0.010049250597053265,
  "max_absolute_error": 0.02712728159564809,
  "relative_error_percent_on_prices_above_0_01": 4.553818228094953,
  "train_n": 1000,
  "validation_n": 200,
  "test_n": 500,
  "best_epoch": 464,
  "training_seconds": 14.46550431300011,
  "parameter_count": 34433,
  "median_batch_inference_ms_500": 0.4863609999574692,
  "median_inference_micro